[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C46_Graph_ML_Course/01_graph_basics/01_graph_basics.ipynb)

# 01 · 图与谱基础（用 numpy 从零）

目标：把图彻底**矩阵化**，并从**图拉普拉斯的谱**里读出图的结构。全部 numpy 从零、可对拍。

路线：邻接/度/拉普拉斯 → 心脏恒等式 → 三种归一化 → 谱与连通分量 → **Fiedler 向量谱聚类(SBM)** → 随机游走平稳分布 / PageRank → ✏️ 练习 → 📖 答案 → 🧪 Karate club 真实图胶囊。

> 心智模型：**L = D − A 是图上的二阶差分；它的特征向量是图的傅里叶基；λ₂(谱隙) 告诉你图多难切开。**

## 1 · 邻接、度、拉普拉斯

三块基石：邻接矩阵 `A`、度矩阵 `D`（对角，行和）、组合拉普拉斯 `L = D − A`。
用一个 6 节点、两个三角形 + 一条桥的小图（两个明显社区）。

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

def build_graph(edges, n):
    A = np.zeros((n, n))
    for i, j in edges:
        A[i, j] = A[j, i] = 1.0      # 无向
    return A

# 三角形 {0,1,2} + 三角形 {3,4,5} + 桥 2-3
edges = [(0,1),(1,2),(0,2),(3,4),(4,5),(3,5),(2,3)]
n = 6
A = build_graph(edges, n)
deg = A.sum(1)
D = np.diag(deg)
L = D - A
print('A=\n', A)
print('度 deg =', deg)
print('L = D - A =\n', L)
assert np.allclose(A, A.T)
assert np.allclose(L.sum(1), 0), 'L 每行和必为 0（常向量是零特征向量）'
print('✅ L 每行和为 0 —— 这保证常向量 1 是特征值 0 的特征向量')

## 2 · 心脏恒等式：xᵀLx = ½ Σ_(i,j)∈E (x_i − x_j)²

这是整个谱图理论的核心。它说 L 的二次型 = 所有边两端取值之差的平方和（的一半）。
推论：L 半正定（二次型 ≥ 0），特征值全部非负。我们对随机信号逐项验证。

In [ ]:
rng = np.random.default_rng(0)
x = rng.standard_normal(n)

quad = x @ L @ x                       # 左边：二次型
edge_sum = 0.5 * sum((x[i] - x[j])**2 for i, j in edges) * 2  # 注意 edges 已是每条边一次
# edges 列表每条无向边只出现一次，所以差平方和直接累加即可（不再乘 1/2 再 *2）
edge_sum = sum((x[i] - x[j])**2 for i, j in edges)
print(f'xᵀLx              = {quad:.6f}')
print(f'Σ_(i,j)∈E (xi-xj)² = {edge_sum:.6f}')
assert np.allclose(quad, edge_sum), '心脏恒等式应当成立'
# 半正定：随便来一堆向量，二次型都 >= 0
for _ in range(100):
    v = rng.standard_normal(n)
    assert v @ L @ v >= -1e-9
print('✅ 心脏恒等式成立，且 L 半正定（100 个随机向量二次型全 ≥ 0）')

## 3 · 三种拉普拉斯与归一化

组合型 `L = D − A`；对称归一化 `L_sym = I − D^{-1/2} A D^{-1/2}`（**GCN 的出发点**）；
随机游走型 `L_rw = I − D^{-1} A`。验证：L_sym 与 L_rw **特征值相同**（相似变换），L_sym 特征值落在 [0, 2]。

In [ ]:
def laplacians(A):
    deg = A.sum(1)
    D = np.diag(deg)
    L = D - A
    dinv_sqrt = np.diag(1.0 / np.sqrt(deg))
    dinv = np.diag(1.0 / deg)
    L_sym = np.eye(len(A)) - dinv_sqrt @ A @ dinv_sqrt
    L_rw  = np.eye(len(A)) - dinv @ A
    return L, L_sym, L_rw

L, L_sym, L_rw = laplacians(A)
ev_sym = np.sort(np.linalg.eigvalsh(L_sym))
ev_rw  = np.sort(np.linalg.eigvals(L_rw).real)   # L_rw 非对称，用 eigvals
print('L_sym 特征值:', ev_sym)
print('L_rw  特征值:', ev_rw)
assert np.allclose(ev_sym, ev_rw, atol=1e-8), 'L_sym 与 L_rw 特征值应相同'
assert ev_sym.min() > -1e-8 and ev_sym.max() < 2 + 1e-8, 'L_sym 特征值应在 [0,2]'
print('✅ L_sym 与 L_rw 同谱；L_sym 特征值落在 [0,2] —— 这是 GCN 数值稳定的来源')

## 4 · 谱与连通性：零特征值重数 = 连通分量数

谱图理论最漂亮的结论：**L 的特征值 0 的重数 = 图的连通分量个数**；λ₂ > 0 ⟺ 图连通。
我们对比「连通图」与「人为断桥后的两块图」，从谱里把连通分量**数出来**。

In [ ]:
def count_zero_eigs(L, tol=1e-8):
    ev = np.sort(np.linalg.eigvalsh(L))
    return int(np.sum(ev < tol)), ev

# 连通图（有桥 2-3）
k1, ev1 = count_zero_eigs(L)
print('连通图  : 零特征值重数 =', k1, ' 最小两个特征值 =', ev1[:2])

# 断桥：去掉 2-3，变成两个三角形（2 个分量）
edges_cut = [e for e in edges if e != (2,3)]
A_cut = build_graph(edges_cut, n)
L_cut = np.diag(A_cut.sum(1)) - A_cut
k2, ev2 = count_zero_eigs(L_cut)
print('断桥后  : 零特征值重数 =', k2, ' 最小两个特征值 =', ev2[:2])

assert k1 == 1, '连通图应只有 1 个零特征值'
assert k2 == 2, '断成两块应有 2 个零特征值'
assert ev1[1] > 1e-6, '连通图 λ₂ > 0'
assert ev2[1] < 1e-8, '不连通图 λ₂ = 0'
print('✅ 从谱「数出」连通分量：连通→1 个零特征值(λ₂>0)，断桥→2 个零特征值(λ₂=0)')

## 5 · 谱聚类：用 Fiedler 向量切开 SBM 的社区

**随机块模型 (SBM)**：块内连边概率 p 高、块间 q 低，天然有社区。因为我们自己生成，**每个节点真正属于哪个社区是 ground truth**。

谱聚类：取 L_sym 第二小特征向量（**Fiedler 向量**），按符号二分，与 ground truth 对拍。

In [ ]:
def make_sbm(sizes, p_in, p_out, seed=0):
    '''生成 SBM。返回 (A, labels)。labels 是 ground-truth 社区标签。'''
    rng = np.random.default_rng(seed)
    n = sum(sizes)
    labels = np.concatenate([np.full(s, k) for k, s in enumerate(sizes)])
    A = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            prob = p_in if labels[i] == labels[j] else p_out
            if rng.random() < prob:
                A[i, j] = A[j, i] = 1.0
    return A, labels

def fiedler_bipartition(A):
    '''用 L_sym 的第二小特征向量(Fiedler)按符号把图二分。'''
    deg = A.sum(1)
    dinv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(deg, 1e-12)))
    L_sym = np.eye(len(A)) - dinv_sqrt @ A @ dinv_sqrt
    w, V = np.linalg.eigh(L_sym)        # 升序
    fiedler = V[:, 1]                    # 第二小
    return (fiedler > 0).astype(int), w[1]

def cluster_accuracy(pred, true):
    '''二分聚类对拍：标签可能翻转，取两种对齐的较大者。'''
    acc = (pred == true).mean()
    return max(acc, 1 - acc)

A_sbm, labels = make_sbm([15, 15], p_in=0.5, p_out=0.05, seed=1)
pred, lam2 = fiedler_bipartition(A_sbm)
acc = cluster_accuracy(pred, labels)
print(f'SBM(p_in=0.5, p_out=0.05): λ₂={lam2:.4f}, 谱聚类准确率={acc:.2%}')
assert acc >= 0.9, '清晰社区结构下谱聚类应几乎完美'
print('✅ Fiedler 向量按符号二分，几乎完美还原 ground-truth 社区')

**对照实验**：当 `p_in ≈ p_out`（社区结构模糊）时，谱隙 λ₂ 变大、谱聚类失效。验证「结构越清晰越好切」。

In [ ]:
for p_out in [0.05, 0.2, 0.45]:
    A_s, lab = make_sbm([15, 15], p_in=0.5, p_out=p_out, seed=1)
    pr, l2 = fiedler_bipartition(A_s)
    print(f'p_out={p_out:.2f}: λ₂={l2:.3f}, 准确率={cluster_accuracy(pr, lab):.2%}')
# 清晰结构(p_out小) 准确率应高于模糊结构(p_out大)
A_clear, lc = make_sbm([15,15], 0.5, 0.05, seed=1)
A_fuzzy, lf = make_sbm([15,15], 0.5, 0.45, seed=1)
acc_clear = cluster_accuracy(fiedler_bipartition(A_clear)[0], lc)
acc_fuzzy = cluster_accuracy(fiedler_bipartition(A_fuzzy)[0], lf)
assert acc_clear > acc_fuzzy, '社区越清晰，谱聚类越准'
print('✅ 社区越清晰(p_out越小)，谱聚类越准——谱聚类依赖同配的块结构')

## 6 · 随机游走平稳分布与 PageRank

转移矩阵 `P = D^{-1} A`（行随机）。平稳分布 `π P = π`。无向连通图上 **π_i ∝ d_i**（度越大越常被访问）。
用**幂迭代**从零算 π，并验证它正比于度；再加传送项得到 PageRank。

In [ ]:
def power_iteration_stationary(P, n_iter=2000, tol=1e-12):
    '''幂迭代求左特征向量(平稳分布)：π <- π P 直到收敛。'''
    n = P.shape[0]
    pi = np.ones(n) / n
    for _ in range(n_iter):
        new = pi @ P
        new = new / new.sum()
        if np.max(np.abs(new - pi)) < tol:
            pi = new; break
        pi = new
    return pi

# 用模块前面的连通小图
deg = A.sum(1)
P = np.diag(1.0 / deg) @ A
assert np.allclose(P.sum(1), 1.0), 'P 必须行随机'
pi = power_iteration_stationary(P)
print('平稳分布 π     =', pi)
print('度归一化 d/Σd  =', deg / deg.sum())
assert np.allclose(pi, deg / deg.sum(), atol=1e-6), '无向连通图 π_i ∝ d_i'
print('✅ 平稳分布正比于度 —— 这正是 GCN 反复聚合会过平滑的根源')

In [ ]:
def pagerank(A, alpha=0.15, n_iter=2000, tol=1e-12):
    '''PageRank：π = (1-α) π P + α (1/n)。alpha=重启概率。'''
    n = A.shape[0]
    P = np.diag(1.0 / A.sum(1)) @ A
    pi = np.ones(n) / n
    teleport = np.ones(n) / n
    for _ in range(n_iter):
        new = (1 - alpha) * (pi @ P) + alpha * teleport
        if np.max(np.abs(new - pi)) < tol:
            pi = new; break
        pi = new
    return pi / pi.sum()

pr = pagerank(A, alpha=0.15)
print('PageRank        =', pr)
assert np.allclose(pr.sum(), 1.0), 'PageRank 是概率分布'
# alpha->0 时 PageRank -> 纯随机游走平稳分布
pr0 = pagerank(A, alpha=1e-6)
assert np.allclose(pr0, pi, atol=1e-3), 'alpha→0 应退回随机游走平稳分布'
print('✅ PageRank 是合法分布；α→0 退回随机游走平稳分布')

---
## ✏️ 练习 1：从零构建邻接矩阵与度

给定边列表（无向）和节点数 `n`，实现 `adjacency_and_degree(edges, n)`，返回 `(A, deg)`：
`A` 是对称 0/1 邻接矩阵，`deg` 是度向量。要求**幂等**（同一条边出现两次不应重复计数）。

In [ ]:
def adjacency_and_degree(edges, n):
    A = np.zeros((n, n))
    # TODO: 遍历 edges，对每条 (i,j) 置 A[i,j]=A[j,i]=1（用赋值=1 而非 +=1 以幂等）
    #       再算 deg = A.sum(axis=1)
    raise NotImplementedError
    return A, deg

In [ ]:
# —— 练习 1 自测 ——
e = [(0,1),(1,2),(0,1),(2,0)]      # (0,1) 故意重复
A_t, deg_t = adjacency_and_degree(e, 3)
assert np.allclose(A_t, A_t.T), 'A 必须对称'
assert A_t[0,1] == 1.0 and deg_t.tolist() == [2,2,2], '重复边不应重复计数'
print('✅ 练习 1 通过：对称、幂等、度数正确')

## ✏️ 练习 2：对称归一化拉普拉斯

实现 `sym_norm_laplacian(A)` 返回 `L_sym = I − D^{-1/2} A D^{-1/2}`。
用 `np.maximum(deg, 1e-12)` 防止孤立点除零。验证：对称、特征值落在 [0, 2]。

In [ ]:
def sym_norm_laplacian(A):
    # TODO: deg=A.sum(1); dinv_sqrt=diag(1/sqrt(max(deg,1e-12)));
    #       return I - dinv_sqrt @ A @ dinv_sqrt
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
Ls = sym_norm_laplacian(A)
assert np.allclose(Ls, Ls.T), 'L_sym 必须对称'
ev = np.linalg.eigvalsh(Ls)
assert ev.min() > -1e-8 and ev.max() < 2 + 1e-8, '特征值应在 [0,2]'
assert abs(ev.min()) < 1e-8, '最小特征值应为 0'
print(f'✅ 练习 2 通过：L_sym 对称，特征值范围 [{ev.min():.3f}, {ev.max():.3f}] ⊆ [0,2]')

## ✏️ 练习 3：Fiedler 向量谱聚类

实现 `spectral_bipartition(A)`：用**组合**拉普拉斯 `L = D − A` 的第二小特征向量（Fiedler 向量），
按其**中位数**把节点二分（比按 0 切更稳健），返回 0/1 标签数组。在 SBM 上对拍 ground truth。

In [ ]:
def spectral_bipartition(A):
    # TODO: L = diag(A.sum(1)) - A; w,V = eigh(L); f = V[:,1] (第二小)
    #       thresh = median(f); return (f > thresh).astype(int)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
A_ex, lab_ex = make_sbm([20, 20], p_in=0.5, p_out=0.05, seed=7)
pred_ex = spectral_bipartition(A_ex)
assert set(np.unique(pred_ex)) <= {0, 1}
acc_ex = cluster_accuracy(pred_ex, lab_ex)
assert acc_ex >= 0.9, f'清晰 SBM 上谱聚类应 ≥90%%，实际 {acc_ex:.2%}'
print(f'✅ 练习 3 通过：谱聚类准确率 {acc_ex:.2%}')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def adjacency_and_degree(edges, n):
    A = np.zeros((n, n))
    for i, j in edges:
        A[i, j] = 1.0
        A[j, i] = 1.0          # 赋值而非 +=，天然幂等
    deg = A.sum(axis=1)
    return A, deg

In [ ]:
# 练习 2 参考答案
def sym_norm_laplacian(A):
    deg = A.sum(1)
    dinv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(deg, 1e-12)))
    return np.eye(len(A)) - dinv_sqrt @ A @ dinv_sqrt

In [ ]:
# 练习 3 参考答案
def spectral_bipartition(A):
    L = np.diag(A.sum(1)) - A
    w, V = np.linalg.eigh(L)
    f = V[:, 1]                 # Fiedler 向量
    return (f > np.median(f)).astype(int)

---
## 🧪 真实数据胶囊：Zachary's Karate Club 的真实社区分裂

Zachary 空手道俱乐部（1977）是图分析最经典的真实数据：34 个成员，俱乐部因教练(节点0)与会长(节点33)矛盾**真实分裂成两派**。
我们用谱聚类去**预测这次分裂**，与历史记录的真实派系对拍。优先用 networkx 取真实图，失败则回退到内置的真实边列表。

In [ ]:
# 真实 Karate club：优先 networkx，失败回退到内置真实边（Zachary 1977 原始数据）
def load_karate():
    try:
        import networkx as nx
        G = nx.karate_club_graph()
        n = G.number_of_nodes()
        A = np.zeros((n, n))
        for i, j in G.edges():
            A[i, j] = A[j, i] = 1.0
        # networkx 自带每个节点的真实派系 'Mr. Hi' / 'Officer'
        faction = np.array([0 if G.nodes[i]['club'] == 'Mr. Hi' else 1 for i in range(n)])
        return A, faction, 'networkx'
    except Exception:
        # 回退：Zachary 1977 的真实边列表（0-indexed）
        E = [(0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),(0,12),(0,13),
             (0,17),(0,19),(0,21),(0,31),(1,2),(1,3),(1,7),(1,13),(1,17),(1,19),(1,21),
             (1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),(3,7),(3,12),(3,13),
             (4,6),(4,10),(5,6),(5,10),(5,16),(6,16),(8,30),(8,32),(8,33),(9,33),(13,33),
             (14,32),(14,33),(15,32),(15,33),(18,32),(18,33),(19,33),(20,32),(20,33),
             (22,32),(22,33),(23,25),(23,27),(23,29),(23,32),(23,33),(24,25),(24,27),(24,31),
             (25,31),(26,29),(26,33),(27,33),(28,31),(28,33),(29,32),(29,33),(30,32),(30,33),
             (31,32),(31,33),(32,33)]
        n = 34
        A = np.zeros((n, n))
        for i, j in E:
            A[i, j] = A[j, i] = 1.0
        # 真实派系（历史记录）：Mr.Hi 一派 vs Officer 一派
        hi = {0,1,2,3,4,5,6,7,8,10,11,12,13,16,17,19,21}
        faction = np.array([0 if i in hi else 1 for i in range(n)])
        return A, faction, 'builtin'

A_k, faction, src = load_karate()
print(f'Karate club 载入自: {src}, 节点数={len(A_k)}, 边数={int(A_k.sum()//2)}')
assert len(A_k) == 34, 'Karate club 应有 34 个成员'
assert np.allclose(A_k, A_k.T)
print('✅ 真实图载入成功')

**🧪 胶囊练习**：用你在练习 3 写的 `spectral_bipartition` 预测 Karate club 的分裂，与真实派系 `faction` 对拍准确率。
（提示：直接调用即可。历史上谱方法能正确分类 ~33/34 个成员。）

In [ ]:
# 你的任务：预测分裂并算准确率（用 spectral_bipartition + cluster_accuracy）
pred_karate = None  # TODO: spectral_bipartition(A_k)
acc_karate = None   # TODO: cluster_accuracy(pred_karate, faction)
raise NotImplementedError

In [ ]:
# 自测
assert pred_karate is not None and acc_karate is not None
print(f'谱聚类预测 Karate 分裂准确率 = {acc_karate:.2%}')
assert acc_karate >= 0.9, '谱方法应能正确预测 ~90%%+ 的成员归属'
print('✅ 胶囊练习通过：纯谱方法从图结构预测了一场真实的社会分裂！')

In [ ]:
# 📖 胶囊参考答案
pred_karate = spectral_bipartition(A_k)
acc_karate = cluster_accuracy(pred_karate, faction)
print(f'谱聚类预测 Karate 分裂准确率 = {acc_karate:.2%}')
assert acc_karate >= 0.9
print('✅ 纯靠图拉普拉斯的第二小特征向量，就还原了 1977 年那场真实的俱乐部分裂')

### 小结
- **图 = (A, X)**；拉普拉斯 **L = D − A** 是图上的二阶差分，二次型 = 边两端差平方和（半正定）。
- **谱 = 图的傅里叶基**：特征值=频率（小=平滑/大=震荡），特征向量=频率模式。
- **零特征值重数 = 连通分量数**；λ₂(谱隙) 度量图多难切开；**Fiedler 向量做谱聚类**。
- **随机游走 P=D⁻¹A**：平稳分布 π_i ∝ d_i（→ GCN 过平滑的根源）；PageRank 加传送项。

下一站：**模块 02 · 消息传递与 GCN** —— 把这里的对称归一化拉普拉斯，变成一行可学习的传播规则 σ(ÂXW)。